# Lecture 8 Exercises
In this exercise we classify land and water using Sentinel-1 and random forest. The purpose of this exercise is to introduce you to basic synthetic aperture radar data handling and explore the issue of speckle.

### Exercise 1: basic land surface classification with SAR

In [ ]:
!pip install geemap --quiet

In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='earthengine-ml-testing') #<- Remember to change this to your own project's name!

In [ ]:
# Load Sentinel-1 GRD data
region = ee.Geometry.Rectangle([174.6, -36.9, 174.9, -36.6])  # Auckland area
s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(region) \
    .filterDate('2024-07-01', '2024-07-15') \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')) \
    .filter(ee.Filter.eq('resolution_meters', 10)) \
    .select(['VV', 'VH']) \
    .mean()

# Add it to a map object for later display
Map = geemap.Map(center=[-36.8, 174.75], zoom=11)
Map.addLayer(s1, {'min': -20, 'max': 0}, 'S1 VV/VH Composite')

In [ ]:
# Get automatic training data from ESA WorldCover
worldcover = ee.Image("ESA/WorldCover/v200/2021").clip(region)

# ESA WorldCover class codes: 80 = permanent water bodies, 10-100 = various land covers
water = worldcover.eq(80)

# Select only Urban, Grassland, Forest as Land
land = worldcover.eq(10).Or(
    worldcover.eq(30)
).Or(
    worldcover.eq(50))

In [ ]:
# Sample points for each class
water_samples = s1.sampleRegions(
    collection=water.stratifiedSample(
        numPoints=200,
        classBand='Map',
        region=region,
        scale=10,
        seed=42,
        geometries=True
    ),
    scale=10,
    geometries=True
).map(lambda f: f.set('class', 1))

land_samples = s1.sampleRegions(
    collection=land.stratifiedSample(
        numPoints=200,
        classBand='Map',
        region=region,
        scale=10,
        seed=43,
        geometries=True
    ),
    scale=10,
    geometries=True
).map(lambda f: f.set('class', 0))

training_fc = land_samples.merge(water_samples)

In [ ]:
# Train a Random Forest classifier
classifier = ee.Classifier.smileRandomForest(numberOfTrees=50).train(
    features=training_fc,
    classProperty='class',
    inputProperties=['VV', 'VH']
)

In [ ]:
# Classify the image
classified = s1.classify(classifier)

In [ ]:
# Visualize the results
Map.addLayer(worldcover, {'min': 10, 'max': 100, 'palette': ['green', 'brown', 'blue']}, 'ESA WorldCover (reference)')
Map.addLayer(classified, {'min': 0, 'max': 1, 'palette': ['brown', 'blue']}, 'Land/Water Classification')
Map

**Q1**: Finish the code below in order to generate an accuracy assesment of this RF model. What is the accuracy that we get?

In [ ]:
# --- Accuracy Assessment Exercise ---
# Step 1: Split data into training and validation sets
# Hint: use .randomColumn() to add a column with random values, then filter by <0.7 for training and >=0.7 for validation
with_random = training_fc.randomColumn(seed=123, columnName='random')
training_split = with_random.filter(### YOUR CODE HERE ###)  # e.g. filter(ee.Filter.lt(...))
validation_split = with_random.filter(### YOUR CODE HERE ###)

# Step 2: Retrain classifier on training_split only
classifier_split = ee.Classifier.smileRandomForest(numberOfTrees=50).train(
    features=### YOUR CODE HERE ###,
    classProperty='class',
    inputProperties=['VV', 'VH'])

# Step 3: Classify the validation data
validated = validation_split.classify(classifier_split)

# Step 4: Compute confusion matrix and accuracy
conf_matrix = ### YOUR CODE HERE ###  # e.g. validated.errorMatrix(...)
print('Confusion Matrix:', conf_matrix.getInfo())
print('Overall Accuracy:', conf_matrix.accuracy().getInfo())


### Exercise 2: exploring polarization and roughness
In this exercise you will:
- Visualize VV vs VH backscatter.
- Compare how different classes (land vs water) separate in feature space.
- Think about how polarization is linked to surface roughness and target type.

In [ ]:
# --- Feature Space Exploration with Class Labels (Raw vs Filtered) ---
import matplotlib.pyplot as plt
image_to_use = s1

# Sample VV/VH values from validation points (with class labels)
sampled_points = image_to_use.sampleRegions(
    collection=validation_split,
    properties=['class'],
    scale=10
)

# Bring to client side
validation_list = sampled_points.toList(sampled_points.size())
validation_data = validation_list.getInfo()

# Extract VV, VH, and class labels
vv = [f['properties']['VV'] for f in validation_data]
vh = [f['properties']['VH'] for f in validation_data]
labels = [f['properties']['class'] for f in validation_data]

# Split by class
vv_land = [vv[i] for i in range(len(vv)) if labels[i] == 0]
vv_water = [vv[i] for i in range(len(vv)) if labels[i] == 1]
vh_land = [vh[i] for i in range(len(vh)) if labels[i] == 0]
vh_water = [vh[i] for i in range(len(vh)) if labels[i] == 1]

fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# Left: VV Histogram per class
axs[0].hist(vh_land, bins=50, alpha=0.5, label='Land', color='green')
axs[0].hist(vh_water, bins=50, alpha=0.5, label='Water', color='blue')
axs[0].set_xlabel('VH Backscatter (dB)')
axs[0].set_ylabel('Pixel Count')
axs[0].set_title(f'VH Histogram per Class')
axs[0].legend()
axs[0].grid()

# Right: VV vs VH scatter per class
axs[1].scatter(vv_land, vh_land, alpha=0.5, s=10, label='Land', color='green')
axs[1].scatter(vv_water, vh_water, alpha=0.5, s=10, label='Water', color='blue')
axs[1].set_xlabel("VV Backscatter (dB)")
axs[1].set_ylabel("VH Backscatter (dB)")
axs[1].set_title(f"Feature Space: VV vs VH")
axs[1].legend()
axs[1].grid()

plt.tight_layout()
plt.show()


In [ ]:
# --- Polarization Ratio Visualization ---
# Compute VV/VH ratio (in dB space)
vv_vh_ratio = s1.select('VV').subtract(s1.select('VH')).rename('VVVH_ratio')

# Visualization parameters: darker = low ratio (likely water), bright = high ratio (likely land)
ratio_vis = {
    'min': -5,    # Adjust if your scene is mostly bright or dark
    'max': 5,
    'palette': ['blue', 'white', 'brown']
}

Map = geemap.Map(center=[-36.8, 174.75], zoom=11)
Map.addLayer(vv_vh_ratio, ratio_vis, 'VV/VH Ratio (dB)')
Map


**Q2**: Use the code provided above to explore the following questions:

1. Interpretation:
- Do land and water points cluster in different regions of VV–VH space?
- Which polarization (VV or VH) seems to separate land and water better (if at all)?

2. Surface roughness:
- What would you expect to happen to VV and VH backscatter if the water becomes rougher (e.g., windier conditions)?
- How might that affect the classification?

3. Extension challenge:
- Create a new feature band VV/VH (ratio) or VV - VH (difference) and add it to the training data.
- Retrain the model and check if accuracy improves.

Fundamentally, we do not have good class seperation here. What are some of the reasons that could be causing this?

### Exercise 3: dealing with speckle
You should have seen through exercise 1 and 2 that whilst we can fairly easily visually see the difference between water and land in S1. However, when we do it computationally (at least with this basic RF approach anyway) it is harder. One of the reasons for this is the 'speckle' over land resulting in a mixture of pixels that appear darker than they otherwise should.

For our final exercise, let us do something about that:

In [ ]:
# --- Speckle Exploration ---
# Display the raw VV band to show speckle clearly
Map.addLayer(s1.select('VV'), {'min': -20, 'max': 0}, 'VV (Raw with Speckle)')

# Apply a simple speckle filter: focal_mean (boxcar)
# NOTE: Speckle filters smooth the data, but can also blur edges.
# Experiment with different kernel sizes (e.g., radius = 30)
s1_speckle = s1.focal_mean(radius=50, units='meters')  # 50m smoothing kernel

Map = geemap.Map(center=[-36.8, 174.75], zoom=11)
Map.addLayer(s1_speckle.select('VV'), {'min': -20, 'max': 0}, 'VV (Speckle Reduced)')
Map


**Q3**: Re-run the entire classification pipeline with the de-speckled image. Is there a difference?

Bonus fun- a histogram that demonstrates the effect of despeckling, and let's do that vv-vh space plot again!

In [ ]:
# --- VV Histogram + VV–VH Scatter Before/After Speckle Filtering (Fixed) ---

import matplotlib.pyplot as plt

# Sample VV and VH values before and after filtering
sample_points_raw = s1.sample(region=region, scale=10, numPixels=2000, seed=99)
sample_points_filt = s1_speckle.sample(region=region, scale=10, numPixels=2000, seed=99)

# Get client-side dictionaries
raw_data = sample_points_raw.getInfo()['features']
filt_data = sample_points_filt.getInfo()['features']

# Extract VV and VH into Python lists
vv_raw = [f['properties']['VV'] for f in raw_data]
vh_raw = [f['properties']['VH'] for f in raw_data]
vv_filt = [f['properties']['VV'] for f in filt_data]
vh_filt = [f['properties']['VH'] for f in filt_data]

fig, axs = plt.subplots(1, 2, figsize=(12,5))

# Left: VV histograms
axs[0].hist(vv_raw, bins=40, alpha=0.5, label='Raw VV')
axs[0].hist(vv_filt, bins=40, alpha=0.5, label='Filtered VV')
axs[0].set_xlabel('VV Backscatter (dB)')
axs[0].set_ylabel('Pixel Count')
axs[0].set_title('VV Distribution Before/After Speckle Filtering')
axs[0].legend()
axs[0].grid()

# Right: VV vs VH scatter (before and after)
axs[1].scatter(vv_raw, vh_raw, alpha=0.4, s=10, label='Raw', color='red')
axs[1].scatter(vv_filt, vh_filt, alpha=0.4, s=10, label='Filtered', color='blue')
axs[1].set_xlabel('VV (dB)')
axs[1].set_ylabel('VH (dB)')
axs[1].set_title('VV–VH Feature Space Before/After Filtering')
axs[1].legend()
axs[1].grid()

plt.tight_layout()
plt.show()
